# SQL queries

Queries run against the project SQLite DB (`project_config`: `DATA_DIR` / `DB_FILENAME`, usually `data/raw/events.db`).

## Public documentation (GitHub `Event`)

Authoritative definitions of the JSON stored in `events.event_data`:

- [**GitHub REST API — Activity — Events**](https://docs.github.com/en/rest/activity/events) — `Event` resource (top-level keys, `Actor`, `repo`, `payload` as a typed union).
- [**GitHub event types and payloads**](https://docs.github.com/en/webhooks-and-events/events/github-event-types) — each `type` and the shape of `payload`.
- **Nested resources inside `payload`** follow the normal REST objects, e.g. [Pull requests](https://docs.github.com/en/rest/pulls/pulls), [PR reviews](https://docs.github.com/en/rest/pulls/reviews), [Review comments](https://docs.github.com/en/rest/pulls/comments), [Issues](https://docs.github.com/en/rest/issues/issues), [Issue comments](https://docs.github.com/en/rest/issues/comments).
- [**GHArchive**](https://www.gharchive.org/) — hourly archives of public GitHub events (same overall `Event`-style JSON per line).

---

## GitHub `Event` schema (verbatim JSON in `event_data`)

GHArchive lines are GitHub **activity events**: one JSON object per event. The blob is **not** narrowed by this repo; GitHub may add optional fields over time. Below is the **fixed envelope** and **`payload` layout for the event types this pipeline ingests**; anything under `pull_request`, `issue`, `comment`, `review`, or embedded `user` / `label` / … objects is the full REST resource graph described in the links above (open-ended).

### Top-level keys on every `Event`

| Key | Type | Presence | Meaning |
|-----|------|----------|---------|
| `id` | string | required | Unique event id (same value as SQLite `events.id`). |
| `type` | string | required | Event class name, e.g. `PullRequestEvent` (see [event types](https://docs.github.com/en/webhooks-and-events/events/github-event-types)). |
| `actor` | object | required | Account that triggered the event ([`Actor` on the Events API](https://docs.github.com/en/rest/activity/events)). |
| `repo` | object | required | Repository the event refers to (minimal `repo` object on the event, not the full Repository API). |
| `org` | object | optional | Organization context, when applicable; same **shape as `actor`** when present. |
| `payload` | object | required | Type-specific object (discriminated by `type`). |
| `public` | boolean | required | Whether the event is publicly visible. |
| `created_at` | string (ISO 8601) | required | Event timestamp. |

### `actor` and `org` (Event-scoped `Actor`)

These are the **six** fields GitHub documents on the slim `Actor` attached to an `Event` (not the complete [Users](https://docs.github.com/en/rest/users/users) resource):

| Key | Type | Meaning |
|-----|------|---------|
| `id` | integer | Numeric account id. |
| `login` | string | Username / org login. |
| `display_login` | string | Shown when GitHub needs a display form distinct from `login` (may be absent). |
| `gravatar_id` | string \| null | Legacy Gravatar id. |
| `url` | string (URI) | API URL for this user/org. |
| `avatar_url` | string (URI) | Avatar image URL. |

### `repo` on the `Event`

| Key | Type | Meaning |
|-----|------|---------|
| `id` | integer | Numeric repository id. |
| `name` | string | Full name `owner/repo`. |
| `url` | string (URI) | API URL for the repository. |

### `payload` by event type (documented shapes)

Official payload key lists: [**GitHub event types**](https://docs.github.com/en/webhooks-and-events/events/github-event-types) (each section matches `event.type`). Nested objects are full REST resources ([Issues](https://docs.github.com/en/rest/issues/issues), [Issue comments](https://docs.github.com/en/rest/issues/comments), [Pull requests](https://docs.github.com/en/rest/pulls/pulls), [Reviews](https://docs.github.com/en/rest/pulls/reviews), [Review comments](https://docs.github.com/en/rest/pulls/comments)); GHArchive may include extra optional fields GitHub adds over time.

---

#### `IssueCommentEvent`

Documentation: [IssueCommentEvent](https://docs.github.com/en/webhooks-and-events/events/github-event-types#issuecommentevent).

Activity on an **issue or pull request comment** (`action` describes what happened to the comment).

| Payload key | Type | Role |
|-------------|------|------|
| `action` | string | Comment lifecycle, e.g. `created` (see GitHub docs for current values). |
| `issue` | object | The [issue](https://docs.github.com/en/rest/issues/issues#get-an-issue) the comment belongs to (for PRs this is still the issue-shaped API resource). |
| `comment` | object | The [issue comment](https://docs.github.com/en/rest/issues/comments) resource (`body`, `user`, **`author_association`**, timestamps, …). |

This repo reads text from `payload.comment.body` and association from `payload.comment.author_association` (see `GitHubEvent.extract_text_content` / `_get_author_association`).

---

#### `PullRequestEvent`

Documentation: [PullRequestEvent](https://docs.github.com/en/webhooks-and-events/events/github-event-types#pullrequestevent).

Activity on a **pull request** (`action` describes the PR lifecycle or metadata change).

| Payload key | Type | Role |
|-------------|------|------|
| `action` | string | e.g. `opened`, `closed`, `merged`, `reopened`, `assigned`, `unassigned`, `labeled`, `unlabeled`, … |
| `number` | integer | PR number in the repo. |
| `pull_request` | object | The [pull request](https://docs.github.com/en/rest/pulls/pulls#get-a-pull-request) resource (`title`, `body`, `state`, **`author_association`** when present, `head` / `base`, labels, …). |
| `assignee` | object \| omitted | Optional assignee user for assign/unassign actions. |
| `assignees` | array \| omitted | Optional list of assignees. |
| `label` | object \| omitted | Optional label for `labeled` / `unlabeled`. |
| `labels` | array \| omitted | Optional labels on the PR for label actions. |

This repo reads text from `payload.pull_request.title` / `payload.pull_request.body`; association may appear on `payload.pull_request.author_association`.

---

#### `PullRequestReviewEvent`

Documentation: [PullRequestReviewEvent](https://docs.github.com/en/webhooks-and-events/events/github-event-types#pullrequestreviewevent).

Activity on a **pull request review** (approve / comment / request-changes style review).

| Payload key | Type | Role |
|-------------|------|------|
| `action` | string | e.g. `created`, `updated`, `dismissed`. |
| `pull_request` | object | The [pull request](https://docs.github.com/en/rest/pulls/pulls#get-a-pull-request) the review applies to. |
| `review` | object | The [pull request review](https://docs.github.com/en/rest/pulls/reviews#get-a-review-for-a-pull-request) resource (`body`, `state`, **`author_association`**, `user`, `submitted_at`, …). |

This repo reads text from `payload.review.body`; association from `payload.review.author_association`.

---

#### `PullRequestReviewCommentEvent`

Documentation: [PullRequestReviewCommentEvent](https://docs.github.com/en/webhooks-and-events/events/github-event-types#pullrequestreviewcommentevent).

Activity on a **review comment** on the PR diff (line/file comment).

| Payload key | Type | Role |
|-------------|------|------|
| `action` | string | e.g. `created` on the comment. |
| `pull_request` | object | The [pull request](https://docs.github.com/en/rest/pulls/pulls#get-a-pull-request) the comment belongs to. |
| `comment` | object | The [pull request review comment](https://docs.github.com/en/rest/pulls/comments#get-a-review-comment-for-a-pull-request) resource (`body`, `diff_hunk`, `path`, **`author_association`**, `user`, commit SHAs, …). |

This repo reads text from `payload.comment.body`; association from `payload.comment.author_association`.

---

**SQL / code in this repo** typically reads: `json_extract(event_data, '$.repo.name')`, `$.type`, `created_at`, and `author_association` from `$.payload.comment`, `$.payload.review`, `$.payload.pull_request`, or `$.payload.issue` (see `preprocessing.workflow.metadata_from_raw_event` and `judge/storage.py`).


---

## SQLite tables (short)

Full column list: [`docs/DB_SCHEMA.md`](../docs/DB_SCHEMA.md). In short:

| Table | Columns |
|-------|---------|
| `events` | `id`, `event_data` |
| `cleaned` | `id`, `cleaned_text`, `tokens` — join `events` for repo / time / type / association. |
| `scores` | `comment_id`, `model_name`, FUN/NSI/INSI/ISI scores and reasoning pairs, `created_at` |

Judge rubric: [`CONFORMITY_SYSTEM_PROMPT.md`](../docs/papers/CONFORMITY_SYSTEM_PROMPT.md).

---

**Run order:** setup cell → **One sample `event_data`** (markdown + code) → per-table cells (truncated previews).


In [58]:
import json
import sys
from html import escape
from pathlib import Path

import pandas as pd
import sqlite3
from IPython.display import HTML, Markdown, display

# --- configurable sample for deep JSON inspection ---
# Set to a specific GitHub event id string, or None to use the first row in `events`.
SAMPLE_EVENT_ID: str | None = 34516188931

# How much text to show in SQL previews (table peek cells).
EVENT_DATA_PREVIEW_LEN = 200
CLEANED_TEXT_PREVIEW_LEN = 220
TOKENS_PREVIEW_LEN = 140
REASONING_PREVIEW_LEN = 160


def _repo_root_for_notebook() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "project_config.py").is_file():
            return p
    return here


_REPO = _repo_root_for_notebook()
if str(_REPO) not in sys.path:
    sys.path.insert(0, str(_REPO))

from project_config import DATA_DIR, DB_FILENAME, repo_root

DB_PATH = repo_root() / DATA_DIR / DB_FILENAME

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_colwidth", None)


def connect() -> sqlite3.Connection:
    return sqlite3.connect(str(DB_PATH))


def show_sample_event() -> None:
    """Pretty-print one row's `event_data` JSON in a scrollable panel."""
    with connect() as conn:
        if SAMPLE_EVENT_ID:
            row = conn.execute(
                "SELECT id, event_data FROM events WHERE id = ? LIMIT 1",
                (SAMPLE_EVENT_ID,),
            ).fetchone()
        else:
            row = conn.execute("SELECT id, event_data FROM events LIMIT 1").fetchone()
    if not row:
        display(Markdown("_No rows in `events`._"))
        return
    eid, raw = row
    try:
        pretty = json.dumps(json.loads(raw), indent=2, ensure_ascii=False)
    except json.JSONDecodeError:
        pretty = raw[:8000] + ("\n… [truncated]" if len(raw) > 8000 else "")
    display(Markdown(f"##### One sample `event_data` (`id={eid}`)"))
    safe = escape(pretty)
    html = (
        "<div style=\"max-height:28rem;overflow:auto;border:1px solid #ccc;"
        "padding:0.6rem;background:#f9f9f9;font-family:ui-monospace,monospace\">"
        f"<pre style=\"margin:0;white-space:pre-wrap;font-size:12px;line-height:1.35\">{safe}</pre>"
        "</div>"
    )
    display(HTML(html))


def peek_table(name: str) -> None:
    """Row count, PRAGMA layout, and 10 sample rows with truncated wide columns."""
    with connect() as conn:
        display(Markdown("##### Row count"))
        display(pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {name}", conn))
        display(Markdown("##### Columns"))
        display(pd.read_sql_query(f"PRAGMA table_info({name})", conn))
        display(Markdown("##### Sample (10 rows, truncated previews)"))
        if name == "events":
            q = (
                f"SELECT id, substr(event_data, 1, {EVENT_DATA_PREVIEW_LEN}) || '…' AS event_data_preview "
                "FROM events LIMIT 10"
            )
        elif name == "cleaned":
            q = (
                "SELECT id, "
                f"substr(cleaned_text, 1, {CLEANED_TEXT_PREVIEW_LEN}) || '…' AS cleaned_text_preview, "
                f"substr(tokens, 1, {TOKENS_PREVIEW_LEN}) || '…' AS tokens_preview "
                "FROM cleaned LIMIT 10"
            )
        elif name == "scores":
            q = (
                "SELECT comment_id, model_name, "
                "fun_score, nsi_score, insi_score, isi_score, "
                f"substr(fun_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS fun_reasoning_preview, "
                f"substr(nsi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS nsi_reasoning_preview, "
                f"substr(insi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS insi_reasoning_preview, "
                f"substr(isi_reasoning, 1, {REASONING_PREVIEW_LEN}) || '…' AS isi_reasoning_preview "
                "FROM scores LIMIT 10"
            )
        else:
            q = f"SELECT * FROM {name} LIMIT 10"
        display(pd.read_sql_query(q, conn))


print(f"DB: {DB_PATH.resolve()}  exists={DB_PATH.is_file()}")


DB: /Users/naataaniitsosie/repos/swe-principals/data/raw/events.db  exists=True


## One sample `event_data`

Runs `show_sample_event()` using `SAMPLE_EVENT_ID` from the setup cell (`None` = first row in `events`).


In [59]:
show_sample_event()


##### One sample `event_data` (`id=34516188931`)

## `events`

Peek uses truncated `event_data` previews so the notebook stays small. For one full JSON object, run the **One sample `event_data`** cell above.

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM events;` |
| Layout | `PRAGMA table_info(events);` |
| Sample (short) | `SELECT id, substr(event_data,1,200) \|\| '…' AS event_data_preview FROM events LIMIT 10;` |


In [53]:
peek_table("events")


##### Row count

,n
0,41936


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,1
1,1,event_data,TEXT,1,None,0


##### Sample (10 rows, truncated previews)

,id,event_data_preview
0,34509224090,"{""id"": ""34509224090"", ""type"": ""PullRequestEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_url…"
1,34509271007,"{""id"": ""34509271007"", ""type"": ""IssueCommentEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_ur…"
2,34509279118,"{""id"": ""34509279118"", ""type"": ""PullRequestEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_url…"
3,34511613047,"{""id"": ""34511613047"", ""type"": ""PullRequestReviewEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avat…"
4,34511613085,"{""id"": ""34511613085"", ""type"": ""PullRequestReviewEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avat…"
5,34511654797,"{""id"": ""34511654797"", ""type"": ""IssueCommentEvent"", ""actor"": {""id"": 101474753, ""login"": ""salvo-polizzi"", ""display_login"": ""salvo-polizzi"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/salvo-…"
6,34512746838,"{""id"": ""34512746838"", ""type"": ""PullRequestEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_url…"
7,34516188931,"{""id"": ""34516188931"", ""type"": ""PullRequestReviewCommentEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm""…"
8,34516496181,"{""id"": ""34516496181"", ""type"": ""PullRequestReviewEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avat…"
9,34517383060,"{""id"": ""34517383060"", ""type"": ""PullRequestEvent"", ""actor"": {""id"": 2865885, ""login"": ""felixxm"", ""display_login"": ""felixxm"", ""gravatar_id"": """", ""url"": ""https://api.github.com/users/felixxm"", ""avatar_url…"


## `cleaned`

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM cleaned;` |
| Columns | `PRAGMA table_info(cleaned);` |
| Sample | `SELECT * FROM cleaned LIMIT 10;` *(code uses truncated `cleaned_text` / `tokens` previews)* |


In [54]:
peek_table("cleaned")


##### Row count

,n
0,37197


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,id,TEXT,0,None,1
1,1,cleaned_text,TEXT,1,None,0
2,2,tokens,TEXT,1,None,0


##### Sample (10 rows, truncated previews)

,id,cleaned_text_preview,tokens_preview
0,34509224090,fixed #35072 -- corrected field.choices description in models topic. fixed choices description in docs: [ticket](https://code.djangoproject.com/ticket/35072)…,"[""fixed"", ""35072"", ""corrected"", ""field"", ""choices"", ""description"", ""in"", ""models"", ""topic"", ""fixed"", ""choices"", ""description"", ""in"", ""docs"",…"
1,34509271007,"@expert-m thanks for this patch :+1: welcome aboard :boat: > @felixxm please take a look. looks good, thanks :+1:…","[""expert"", ""m"", ""thanks"", ""for"", ""this"", ""patch"", ""1"", ""welcome"", ""aboard"", ""boat"", ""felixxm"", ""please"", ""take"", ""a"", ""look"", ""looks"", ""good…"
2,34509279118,"improved variable names in queryset.delete(). the doc says: ``` the delete method, conveniently, is named [delete()](https://docs.djangoproject.com/en/4.0/ref/models/instances/#django.db.models.model.delete). this method…","[""improved"", ""variable"", ""names"", ""in"", ""queryset"", ""delete"", ""the"", ""doc"", ""says"", ""the"", ""delete"", ""method"", ""conveniently"", ""is"", ""named""…"
3,34511613047,"@salvo-polizzi thanks :+1: i pushed small edits to docs, and copied the same deprecation to the `asave()`.…","[""salvo"", ""polizzi"", ""thanks"", ""1"", ""i"", ""pushed"", ""small"", ""edits"", ""to"", ""docs"", ""and"", ""copied"", ""the"", ""same"", ""deprecation"", ""to"", ""the…"
4,34511613085,"@salvo-polizzi thanks :+1: i pushed small edits to docs, and copied the same deprecation to the `asave()`.…","[""salvo"", ""polizzi"", ""thanks"", ""1"", ""i"", ""pushed"", ""small"", ""edits"", ""to"", ""docs"", ""and"", ""copied"", ""the"", ""same"", ""deprecation"", ""to"", ""the…"
5,34511654797,thanks @felixxm…,"[""thanks"", ""felixxm""]…"
6,34512746838,fixed #35060 -- deprecated passing positional arguments to model.save()/asave(). refs: [ticket 35060](https://code.djangoproject.com/ticket/35060)…,"[""fixed"", ""35060"", ""deprecated"", ""passing"", ""positional"", ""arguments"", ""to"", ""model"", ""save"", ""asave"", ""refs"", ""ticket"", ""35060"", ""https"", ""…"
7,34516188931,"`deduplicate_items` has more limitations, let's leave that to the users.…","[""deduplicate_items"", ""has"", ""more"", ""limitations"", ""let"", ""s"", ""leave"", ""that"", ""to"", ""the"", ""users""]…"
8,34516496181,@ngnpope thanks :+1:…,"[""ngnpope"", ""thanks"", ""1""]…"
9,34517383060,fixed #35075 -- added deduplicate_items parameter to btreeindex. ticket-35075…,"[""fixed"", ""35075"", ""added"", ""deduplicate_items"", ""parameter"", ""to"", ""btreeindex"", ""ticket"", ""35075""]…"


## `scores`

| Step | SQL |
|---:|---|
| Count | `SELECT COUNT(*) AS n FROM scores;` |
| Columns | `PRAGMA table_info(scores);` |
| Sample | `SELECT * FROM scores LIMIT 10;` *(code uses truncated reasoning previews)* |


In [55]:
peek_table("scores")


##### Row count

,n
0,100


##### Columns

,cid,name,type,notnull,dflt_value,pk
0,0,comment_id,TEXT,1,None,1
1,1,model_name,TEXT,1,None,2
2,2,fun_score,INTEGER,1,None,0
3,3,fun_reasoning,TEXT,1,None,0
4,4,nsi_score,INTEGER,1,None,0
5,5,nsi_reasoning,TEXT,1,None,0
6,6,insi_score,INTEGER,1,None,0
7,7,insi_reasoning,TEXT,1,None,0
8,8,isi_score,INTEGER,1,None,0
9,9,isi_reasoning,TEXT,1,None,0


##### Sample (10 rows, truncated previews)

,comment_id,model_name,fun_score,nsi_score,insi_score,isi_score,fun_reasoning_preview,nsi_reasoning_preview,insi_reasoning_preview,isi_reasoning_preview
0,34517387521,gpt-5.4-mini,0,0,0,0,The comment only requests updating a README file and does not indicate any runtime defect or broken behavior.…,It does not explicitly invoke team conventions or a group rule about documentation updates; it is just a terse request.…,"There is no implicit social pressure conveyed beyond the bare request, and no unstated norm can be inferred from the text alone.…","No expert rationale, documentation standard, or technical justification is provided.…"
1,34548195471,gpt-5.4-mini,0,0,0,0,The comment is only a request to update documentation; it does not indicate a runtime or correctness issue in the code.…,"There is no explicit appeal to team norms, project conventions, or belonging. It simply asks for a README update.…",The request is direct but gives no reasoning that would imply an unstated social rule or hidden expectation beyond the literal task.…,"No technical justification, documentation reference, or expert rationale is provided—just a bare instruction to update the README.…"
2,34548363302,gpt-5.4-mini,0,0,0,0,"The comment is a terse request to change documentation content, with no indication of a broken feature or correctness issue.…","It does not explicitly invoke team norms, project conventions, or an expectation about how the group works.…","Although it is an imperative, there is no implied social disapproval or unstated rule beyond the request itself; it reads like a simple suggestion to edit docum…","No expert rationale, documentation citation, or technical justification is provided beyond the instruction to update the README.…"
3,34553639894,gpt-5.4-mini,0,0,0,0,The comment requests updating documentation only; it does not identify a runtime bug or functional correctness issue.…,It does not invoke any team or project norm explicitly; it is just a brief instruction to change a file.…,"There is no implicit social pressure beyond a bare request, and no unwritten rule is suggested by the text alone.…","It does not cite expert guidance, documentation standards, or any technical rationale for why the README should be updated.…"
4,34566005792,gpt-5.4-mini,0,0,0,0,The comment is only a directive to update documentation and does not claim any runtime bug or broken functionality.…,"It does not explicitly reference team conventions, project norms, or a social expectation of belonging.…","Although it is a terse instruction, it does not convey an implicit social rule or disapproval beyond the request itself.…","No expert rationale, documentation standard, or technical justification is provided.…"
5,34571715220,gpt-5.4-mini,0,0,1,1,"The comment does not point to a bug, broken behavior, or objective correctness issue. It is about release planning and deferring further changes to a future ver…","The commenter is urging the team to move forward with a release and scope future requests to v6, but they do not explicitly invoke a project rule or team norm s…","The comment expresses a preference to keep the PR as-is and cut a release, but the social expectation is not grounded in an explicit norm; it is more of an unst…","There is a mild expert-practice argument for shipping developed improvements and deferring additional work to a later major version, but it is framed as opinion…"
6,34571730688,gpt-5.4-mini,0,0,2,1,The comment is not about a concrete bug or broken functionality; it is about release planning and scope control.…,"The reviewer is expressing a preference for project direction, but does not explicitly invoke a team rule or social norm about how contributors should behave.…","The comment pressures the author to stop requesting more changes and to accept the current state, but it does so without explicitly stating the underlying norm.…","The rationale is strategic rather than expert-technical: it argues for cutting a release and deferring extr